![Foundations of NLP] Embeddings()

---



### About:  
Computers are only able to work with numerical data. One of the most important tasks for working with language is convert the text data into a numerical form. Text embeddings aim to preserve the meaning and context of words by assigning a vector to words where similar words with similar meanings or contexts would have similar vectors. In this notebook we will explore progressively more complex text embeddings and talk about how they are used. 


### Objectives:
1. Describe word2vec and cosine similarity 
1. Apply word2vec to customer reviews
1. Evaluate word2vec output
1. Describe the problem contextual embeddings aim to address 
1. Apply BERT model to a dataset and explore outputs of contextual embeddings  


### Prerequisites:
- Python 
- Core data science libraries Numpy and Pandas 
- Understand, run, and evaluate classification models in Scikit Learn 

### Installs
- [gensim](https://radimrehurek.com/gensim/intro.html#installation)
- [pyTorch](https://pytorch.org/get-started/locally/)



### Imports

In [ ]:
#imports 
import numpy as np
import pandas as pd

## Gensisim implementation of word2vec imports 
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import  Word2Vec
import re # regular expression library 

# Required NLTK data downloads 
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

## Pytorch imports for BERT embeddings
import torch
from transformers import BertTokenizer, BertModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


### NLP Scenario 
You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product. We need to draft a report for our client that reports the sentiment of the reviews. 

#####  Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."


In [ ]:
# data setup 
# create a list of reviews 
reviews = ["I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!",
"I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.",
"The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
"I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
"The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.",
"The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."
]

In [ ]:
# Create a function to preprocess the text
def preprocess_text(text):
    """
    Clean and tokenize text
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    
    return tokens

# Preprocess the reviews
processed_reviews = [preprocess_text(review) for review in reviews]

In [ ]:
#lets review our tokenized reviews
for review in processed_reviews:
    print(review)


Compare the tokenized reviews with the original reviews below. Note that the text has been converted to lowercase, special characters and numbers have been removed, and stopwords have been removed.

#####  Original Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."  

### Word2Vec

Text embeddings aim to preserve the meaning and context of words by assigning a vector to words where similar words with similar meanings or contexts would have similar vectors.

<img src="./assets/food-vector-example.png" alt="Word2Vec Example" width="500" height="300">


Notice that angles between words can be used to see how similar two items are to each other. Ice cream and sorbet are very similar to each other, while salad and coffee are very different from one another. In mathematical terms, we can measure these differences using a cosine similarity.   

Cosine similarity ranges in values from -1 to 1. Where 1 is an angle (θ) that is close to 0° or perfectly aligned (ice cream and sorbet are close in our example). In contrast a value of -1 is 180° or opposite each other (coffee and salad in our example). 

Let's code this for our scenario. 


In [ ]:
#initialize the word2vec model on our reviews
model = Word2Vec(sentences=processed_reviews, vector_size=100, window=5, sg=0, min_count=1)

In [ ]:
# Get the word vector for a word (impressed)
model.wv['impressed']

In [ ]:
model.wv['terrible']

In [ ]:
# find the most similar words to a word
model.wv.most_similar('impressed')

In [ ]:
model.wv.most_similar('terrible')

Notice that the word 'impressed' is similar to 'purchasing' and 'love', while the word 'terrible' is similar to 'hype' and 'disappointing'. 

This shows that the Word2Vec model has learned the some of the semantic relationships between words based on the context in which they appear in the reviews. Given our very small training set this is rather impressive. With more data and more examples of usage these would improve. 

### Understanding word order 

Let’s look at the following example:

- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"



## Try it!  
Use the steps above to convert these to vectors using word to vec 

In [ ]:
# create a list of sentences
sentences = ["The supplier agreed to pay the manufacturer",
             "The manufacturer agreed to pay the supplier"]


# Preprocess the sentences using the function we created earlier, preprocess_text()
processed_reviews = [preprocess_text(sentence) for sentence in sentences]

# Initialize the Word2Vec model
model = Word2Vec(sentences=processed_reviews, vector_size=100, window=5, sg=0, min_count=1)


In [ ]:
# Get the word vector for a word (supplier)
model.wv['supplier']

In [ ]:
# Get the word vector for a word (manufacturer)
model.wv['agreed']

#### What did you notice? 

These two sentences would have identical representations in word2vec but have very different meanings. In this example, the order of the words is critical to the meaning of the phrase. We aren't able to capture this order with the bag-of-words approach used in word2vec and other similar models. To address this issue, methods were developed to capture the sequential order of words. 


### Contextual Embeddings 

Unlike word2vec, contextual embeddings create different embeddings for the same word depending on the context in which it was used. To capture the context we use information such as the words around our word of interest, its sentence position, and other factors to map its vector. In other words, contextual embeddings are position aware. 

Let's look at our example from before: 
- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"

We will use BERT, a famous language model that uses transformer encoder architecture, to create contextual embeddings.  
We can access this model using [pyTorch transformers](https://pytorch.org/hub/huggingface_pytorch-transformers/) . 



In [ ]:
# Example sentences
sentence1 = "The supplier agreed to pay the manufacturer"
sentence2 = "The manufacturer agreed to pay the supplier"

In [ ]:
# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

The first step is to create our tokens 

In [ ]:
# Add special tokens and convert to tensor 
input1 = tokenizer(sentence1, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input1['input_ids'][0])

Notice that the BERT tokenizer added two special tokens `[CLS]` is BERT's way of marking the start of the input document (a sentence in this case). `[SEP]` indicates a separation (such as between sentences) and the end of a document. 

## Try it! 
Create the tokens for sentence 2

In [ ]:
# Add special tokens and convert to tensor 
input2 = tokenizer(sentence2, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input2['input_ids'][0])

Let's create a few functions to create the tokens and then use them to create the BERT embeddings at the sentence level and the token level. For now, focus on the outputs. We will dive into how to the details of how code block below works at another time. 

In [ ]:
# Sentence level embeddings
def get_bert_embedding(sentence):
    """
    Get BERT embeddings for a sentence
    """
    # Add special tokens and convert to tensor
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    
    # Get BERT embeddings
    with torch.no_grad(): # No need to calculate gradients. This will save memory and time 
        outputs = model(**inputs)
        
    # Get the embeddings from the last hidden state
    last_hidden_states = outputs.last_hidden_state
    
    # Return both the sentence-level embedding (CLS token) and token-level embeddings
    return {
        'sentence_embedding': last_hidden_states[0][0].numpy(),  # CLS token
        'token_embeddings': last_hidden_states[0].numpy(),      # All tokens
        'tokens': tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    }

Now we can pass each of our sentences to the function to get the embeddings for each. 

In [ ]:
#print each sentence as a reminder of what we are working with
print(sentence1)
print(sentence2)

In [ ]:
# Get BERT embeddings for the sentences
embedding1 = get_bert_embedding(sentence1)
embedding2 = get_bert_embedding(sentence2)

In [ ]:
# Calculate the cosine similarity between the sentence embeddings
similarity = cosine_similarity([embedding1['sentence_embedding']], [embedding2['sentence_embedding']])[0][0]
print(f"Sentence Level Cosine Similarity: {similarity}")

Notice that our two sentences are very similar. This is expected because they contain the exact same words! But let's look at the embeddings for the individual tokens (words in this case). 

Interesting they are no longer perfectly similar. Let's look at the individual embeddings to compare the word 'supplier' in both sentences. 

In [ ]:
#Let's compare the embeddings for supplier in both sentences
#sentence 1
id1 = embedding1['tokens'].index('supplier')
print("Sentence 1")
print(embedding1['token_embeddings'][id1][:10]) # print only first 10 values of the embedding
print("")

#sentence 2
id2 = embedding2['tokens'].index('supplier')
print("Sentence 2")
print(embedding2['token_embeddings'][id2][:10]) # print only first 10 values of the embedding

Same word but different embeddings! That's because the order in which they were used (sequence) was used in the embedding. Below we can see the cosine similarity for both versions of supplier. 

In [ ]:
supplier_similarity = cosine_similarity(
            embedding1['token_embeddings'][id1].reshape(1, -1),
            embedding2['token_embeddings'][id2].reshape(1, -1)
        )[0][0]
print(f"Similarity between 'supplier' in both sentences: {supplier_similarity}")

## Try it! 
Checkout the embeddings for another word of your choice. 

In [ ]:
#Let's compare the embeddings for 'pay' in both sentences
#sentence 1
id1 = embedding1['tokens'].index('pay')
print("Sentence 1")
print(embedding1['token_embeddings'][id1][:10]) # print only first 10 values of the embedding
print("")

#sentence 2
id2 = embedding2['tokens'].index('pay')
print("Sentence 2")
print(embedding2['token_embeddings'][id2][:10]) # print only first 10 values of the embedding
print("")

pay_similarity = cosine_similarity(
            embedding1['token_embeddings'][id1].reshape(1, -1),
            embedding2['token_embeddings'][id2].reshape(1, -1)
        )[0][0]
print(f"Similarity between 'pay' in both sentences: {pay_similarity}")

Notice that "pay" has a very high similarity across the two sentences. 

### Summary
In this notebook, we learned how to: 
1. Use word embeddings to represent text data
2. Use Word2Vec model to generate word embeddings for a list of reviews
   - Word2Vec is a static embedding that has the same numerical representation for a given word regardless of context 
3. Use pre-trained BERT model to generate contextual embeddings for a pair of sentences.
   - BERT is a contextual embedding that creates different embeddings for the same word depending on how it was used in a sentence. 

### Recommended readings 

#### Primary sources 
- [Foundational word2vec paper](https://arxiv.org/abs/1301.3781)  
- [Foundational BERT paper](https://arxiv.org/pdf/1810.04805) 

#### Other helpful references 
tktk

### 